## Without web search

In [7]:
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
import httpx
from langchain_groq import ChatGroq

# Disable SSL verification to work around corporate proxy/firewall
client = httpx.Client(verify=False)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, http_client=client)

In [9]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm
)

In [4]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How up to date is your training knowledge?")

response = agent.invoke(
    {"messages": [question]}
)

In [5]:
print(response['messages'][-1].content)

My knowledge cutoff is currently December 2023, but I have access to more recent information via internet search.


## Add web search tool

In [11]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()
# Disable SSL verification to work around corporate proxy/firewall
tavily_client.session.verify = False

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of San Francisco?")

/Users/L107127/Library/CloudStorage/OneDrive-EliLillyandCompany/Desktop/Foundation-Introduction-to-LangGraph---Python/lc-academy-env/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.tavily.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


{'query': 'Who is the current mayor of San Francisco?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://apnews.com/article/san-francisco-new-mayor-liberal-city-81ea0a7b37af6cbb68aea7ef5cc6a4f0',
   'title': "San Francisco's new mayor is starting to unite the fractured city",
   'content': 'San Francisco Mayor Daniel Lurie, a political newcomer and Levi Strauss heir, has marked his first 100 days with a hands-on, business-friendly approach.',
   'score': 0.9979966,
   'raw_content': None},
  {'url': 'https://en.wikipedia.org/wiki/Mayor_of_San_Francisco',
   'title': 'Mayor of San Francisco - Wikipedia',
   'content': 'The most recent election for a full mayoral term occurred in 2024. In 2022, San Francisco voters passed Proposition H, which changed mayoral elections to the',
   'score': 0.9923638,
   'raw_content': None},
  {'url': 'https://www.cbsnews.com/sanfrancisco/news/san-francisco-mayor-elect-daniel-lurie-claims-victory-outlines-vision-

In [15]:
# Use Llama 4 Scout for better tool-calling support
tool_llm = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0, http_client=client)

agent = create_agent(
    model=tool_llm,
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of San Francisco?")

response = agent.invoke(
    {"messages": [question]}
)

/Users/L107127/Library/CloudStorage/OneDrive-EliLillyandCompany/Desktop/Foundation-Introduction-to-LangGraph---Python/lc-academy-env/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'api.tavily.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [16]:
from pprint import pprint

pprint(response['messages'])

[HumanMessage(content='Who is the current mayor of San Francisco?', additional_kwargs={}, response_metadata={}, id='8a120ab4-f274-4210-b2d8-058380b938e7'),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'qmkw7dehk', 'function': {'arguments': '{"query":"current mayor of San Francisco"}', 'name': 'web_search'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 713, 'total_tokens': 746, 'completion_time': 0.084989385, 'completion_tokens_details': None, 'prompt_time': 0.023622728, 'prompt_tokens_details': None, 'queue_time': 0.044148221, 'total_time': 0.108612113}, 'model_name': 'meta-llama/llama-4-scout-17b-16e-instruct', 'system_fingerprint': 'fp_37da608fc1', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cbf79-e0c0-7761-a2b9-3a3d7c2548dd-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current mayor of San Francisco'}, 'id': 'qmkw7dehk', 't

trace: https://smith.langchain.com/public/59432173-0dd6-49e8-9964-b16be6048426/r